In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from rope_dev_tools.validation.checks.satellite_orbit_density import _orbit_average
from rope_dev_tools.validation.statistics import compute_statistics, format_statistics_text, uncertainty_of_mean

DATA_DIR = '../../../rope-framework/data/models/wam-borealis-v1'
STATISTICS = ['bias', 'rmse', 'std']

## Average Density vs Time

In [ ]:
data = pd.read_csv(f'{DATA_DIR}/validation_data/Average Density.csv', parse_dates=['datetime'])
has_uncert = 'model_uncert' in data.columns and data['model_uncert'].notna().any()
periods = list(dict.fromkeys(data['period']))
altitudes_km = sorted(data['alt_km'].unique(), reverse=True)

def _doy_formatter(show_year):
    def _format(value, _pos=None):
        dt = mdates.num2date(value)
        doy = dt.timetuple().tm_yday
        return f'{dt.year}-{doy:03d}' if show_year else str(doy)
    return mticker.FuncFormatter(_format)

for period in periods:
    period_rows = data[data['period'] == period]
    start_deltas = sorted(period_rows['start_delta'].unique())
    n_deltas = len(start_deltas)
    widest_delta = min(start_deltas)
    n = len(altitudes_km)

    fig, axes = plt.subplots(n, 1, figsize=(10, 4 * n), squeeze=False)
    for ax, alt_km in zip(axes[:, 0], altitudes_km):
        alt_rows = period_rows[period_rows['alt_km'] == alt_km]
        truth_rows = alt_rows[alt_rows['start_delta'] == widest_delta].sort_values('datetime')
        ax.plot(truth_rows['datetime'], truth_rows['truth_density'], label='WAM', linewidth=1.8)

        stats_lines = []
        for delta in start_deltas:
            delta_rows = alt_rows[alt_rows['start_delta'] == delta].sort_values('datetime')
            dl = f'ROPE-WAM-V1 (\u0394{delta:+d}h)' if n_deltas > 1 else 'ROPE-WAM-V1'
            line, = ax.plot(delta_rows['datetime'], delta_rows['model_density'], label=dl, linewidth=1.8)
            if has_uncert:
                y = delta_rows['model_density'].to_numpy()
                u = delta_rows['model_uncert'].to_numpy()
                ax.fill_between(delta_rows['datetime'], y - u, y + u,
                                color=line.get_color(), alpha=0.2, linewidth=0)
            stats = compute_statistics(delta_rows['model_density'].to_numpy(),
                                       delta_rows['truth_density'].to_numpy(), STATISTICS)
            if stats:
                text = format_statistics_text(stats)
                if n_deltas > 1:
                    text = '\n'.join(f'\u0394{delta:+d}h {l}' for l in text.split('\n'))
                stats_lines.append(text)

        ax.set_title(f'{alt_km} km', fontsize=15)
        ax.set_ylabel('kg/m3', fontsize=13)
        ax.tick_params(axis='both', labelsize=11)
        ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.6)
        ax.legend(loc='upper right', fontsize=11)
        ax.xaxis.set_major_formatter(_doy_formatter(show_year=True))
        if stats_lines:
            ax.text(0.02, 0.95, '\n'.join(stats_lines), transform=ax.transAxes, va='top', fontsize=11,
                    bbox={'boxstyle': 'round', 'facecolor': 'white', 'alpha': 0.7})

    axes[-1, 0].set_xlabel('day of year', fontsize=13)
    if n > 1:
        fig.suptitle(f'Average Density \u2014 {period}', fontsize=16)
    fig.tight_layout()
    plt.show()

## Fixed Height Density (lon/lat snapshots & animations)

In [ ]:
from rope_dev_tools.validation.data_artifacts import load_npz
from pathlib import Path

npz_dir = Path(DATA_DIR) / 'validation_data'
snapshot_files = sorted(p.name for p in npz_dir.glob('Fixed Height Density_snapshots_*.npz'))

for fname in snapshot_files:
    npz = load_npz(DATA_DIR, f'validation_data/{fname}')
    times = [str(t) for t in npz['times']]
    lat_range = (float(npz['lat_min_deg']), float(npz['lat_max_deg']))
    lon_range = (float(npz['lon_min_deg']), float(npz['lon_max_deg']))
    phys_frames = list(npz['physics_density'])
    rope_frames = list(npz['rope_density'])
    all_vals = np.concatenate([np.asarray(g).ravel() for g in phys_frames + rope_frames])
    vmin, vmax = float(all_vals.min()), float(all_vals.max())

    n_snaps = len(times)
    fig, axes = plt.subplots(2, n_snaps, figsize=(3 * n_snaps, 6), squeeze=False)
    for col, (t, pg, rg) in enumerate(zip(times, phys_frames, rope_frames)):
        for row, (grid, model_name) in enumerate([(pg, 'physics'), (rg, 'rope')]):
            ax = axes[row, col]
            im = ax.imshow(np.asarray(grid).T, origin='lower', aspect='auto', vmin=vmin, vmax=vmax,
                           extent=[lon_range[0], lon_range[1], lat_range[0], lat_range[1]])
            ax.set_title(f'{model_name} {t}', fontsize=8)
            ax.tick_params(labelsize=7)
            if col == 0:
                ax.set_ylabel('Latitude (deg)', fontsize=8)
    fig.colorbar(im, ax=axes.ravel().tolist(), label='density', shrink=0.8)
    fig.suptitle(fname.replace('.npz', ''), fontsize=10)
    fig.tight_layout()
    plt.show()

## Satellite Track Density

In [ ]:
data = pd.read_csv(f'{DATA_DIR}/validation_data/Satellite Track Density.csv', parse_dates=['datetime'])
periods = list(dict.fromkeys(data['period']))

fig, axes = plt.subplots(len(periods), 1, figsize=(10, 4 * len(periods)), squeeze=False)

for ax, label in zip(axes[:, 0], periods):
    label_rows = data[data['period'] == label]
    orbit_averaged = bool(label_rows['orbit_averaged'].iat[0])
    has_uncert = label_rows['rope_uncert'].notna().any()
    start_deltas = sorted(label_rows['start_delta'].unique())
    n_deltas = len(start_deltas)
    widest_delta = min(start_deltas)

    widest = label_rows[label_rows['start_delta'] == widest_delta]
    if orbit_averaged:
        widest = _orbit_average(widest, label)
    ax.plot(widest['datetime'], widest['physics_density'], label='physics_model', linewidth=1.8)

    stats_lines = []
    for delta in start_deltas:
        delta_rows = label_rows[label_rows['start_delta'] == delta]
        if orbit_averaged:
            extra_agg = {'rope_uncert': lambda s: uncertainty_of_mean(s.to_numpy())} if has_uncert else None
            delta_rows = _orbit_average(delta_rows, label, extra_agg=extra_agg)
        dl = f'rope_model (\u0394{delta:+d}h)' if n_deltas > 1 else 'rope_model'
        line, = ax.plot(delta_rows['datetime'], delta_rows['rope_density'], label=dl, linewidth=1.8)
        if has_uncert:
            y = delta_rows['rope_density'].to_numpy()
            u = delta_rows['rope_uncert'].to_numpy()
            ax.fill_between(delta_rows['datetime'], y - u, y + u,
                            color=line.get_color(), alpha=0.2, linewidth=0)
        stats = compute_statistics(delta_rows['rope_density'].to_numpy(),
                                   delta_rows['physics_density'].to_numpy(), STATISTICS)
        if stats:
            text = format_statistics_text(stats)
            if n_deltas > 1:
                text = '\n'.join(f'\u0394{delta:+d}h {l}' for l in text.split('\n'))
            stats_lines.append(text)

    ax.set_title(label, fontsize=15)
    ax.set_ylabel('density', fontsize=13)
    ax.tick_params(axis='both', labelsize=11)
    ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.6)
    ax.legend(loc='upper right', fontsize=11)
    if stats_lines:
        ax.text(0.02, 0.95, '\n'.join(stats_lines), transform=ax.transAxes, va='top', fontsize=11,
                bbox={'boxstyle': 'round', 'facecolor': 'white', 'alpha': 0.7})

fig.suptitle('Satellite Track Density', fontsize=16)
fig.tight_layout()
plt.show()

## Harmonic FFT

In [ ]:
HARMONIC_FREQS = [1/24, 1/12, 1/8, 1/6]  # diurnal, semidiurnal, terdiurnal, quarterdiurnal
HARMONIC_LABELS = ['24h', '12h', '8h', '6h']

for scan_type in ['altitude_scan', 'latitude_scan']:
    npz_files = sorted((Path(DATA_DIR) / 'validation_data').glob(f'jan2023_harmonic_test_*_{scan_type}.npz'))
    for npz_path in npz_files:
        npz = load_npz(DATA_DIR, f'validation_data/{npz_path.name}')
        freqs = npz['freqs']
        phys_mag = npz['physics_magnitude']
        rope_mag = npz['rope_magnitude']

        if scan_type == 'altitude_scan':
            names = [f'{v:g}km' for v in npz['altitudes_km']]
            suptitle = f'Altitude scan (lat={float(npz["lat_deg"])}\u00b0, lon={float(npz["lon_deg"])}\u00b0)'
        else:
            names = [f'lat={v:g}\u00b0' for v in npz['lats_deg']]
            suptitle = f'Latitude scan (alt={float(npz["alt_km"])}km, lon={float(npz["lon_deg"])}\u00b0)'

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        for ax, mag, title in [(axes[0], phys_mag, 'physics'), (axes[1], rope_mag, 'rope')]:
            for i, name in enumerate(names):
                ax.semilogy(freqs[i], mag[i], label=name, linewidth=1.5)
            for freq, hlabel in zip(HARMONIC_FREQS, HARMONIC_LABELS):
                ax.axvline(freq, color='grey', linestyle='--', linewidth=0.7, alpha=0.5)
                ax.text(freq, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 1, hlabel,
                        ha='center', va='bottom', fontsize=8, color='grey')
            ax.set_title(title, fontsize=14)
            ax.set_xlabel('frequency (hr$^{-1}$)', fontsize=12)
            ax.set_ylabel('|FFT|', fontsize=12)
            ax.legend(fontsize=9)
            ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.6)

        fig.suptitle(suptitle, fontsize=14)
        fig.tight_layout()
        plt.show()